# 論文實驗全自動批改 Pipeline — V4（GitHub 可攜版）

本 Notebook 實作「三層 SLM 批改 → 雙次執行差異比對 → LLM(Claude)專家仲裁 → 加權分數整合」的完整批改管線，物件導向封裝、具備多層防錯與 Fallback 保底機制。

本版本已將原本寫死在程式碼中的**本機絕對路徑**與**API 金鑰**移除，改為可攜式相對路徑與互動輸入，方便從 GitHub clone 下來後直接使用，不需要修改程式碼即可執行（金鑰除外，每次執行需自行輸入）。

## 使用前準備
1. 確認以下資料夾與本 Notebook 放在**同一層目錄**：`題目/`、`學生答案/`、`Promt/`（內含 `SLM1.py`、`SLM2.py`、`SLM3.py`、`LLM仲裁.py`）。首次執行後會自動建立 `評分結果/` 輸出目錄。
2. 執行「全域配置區」下方 cell 前，先將 `STUDENTS` 改成 `學生答案/` 資料夾內實際存在的學生答案檔名（不含 `.txt`）。
3. 準備好 Colab 端 Ollama 的 ngrok Endpoint，以及一組有效的 Anthropic API Key（[申請頁面](https://console.anthropic.com/)），執行到對應 cell 時會請你手動輸入，兩者皆不會寫入檔案或顯示於輸出。

## Pipeline 五大階段
1. 前處理與防錯切分（Preprocessing）
2. 批改雙次執行（SLMs Execution，Round1 / Round2）
3. 實質分數差異過濾（Divergence Analysis）
4. 題為單位之 LLM 仲裁（Arbitration）
5. 分數整合、公式加權與退路（Finalization）

## 輸出成果
每位學生的 `評分結果/{student}/{timestamp}/` 目錄下會產出：
`round1.json`、`round2.json`、`差異分析.json`、`llm_仲裁結果.json`、`最終評分.json`

In [ ]:
# 首次於本機執行前，請確認已安裝下列套件（Colab 端另行安裝 ollama 相關套件）
%pip install anthropic requests --break-system-packages

In [ ]:
import os
import re
import json
import time
import getpass
import logging
import importlib.util
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List, Any, Optional, Tuple

import requests
from anthropic import Anthropic

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('GradingPipelineV4')

## 全域配置區（Configuration Cell）

路徑、學生名單、動態 Prompt 檔名映射、模型與溫度皆集中於此，方便未來更換版本時程式完全解耦。**API 金鑰不在此設定**，請見下方「設定 Anthropic API Key」區塊。

In [ ]:
# BASE_DIR 預設為 Notebook 所在目錄（相對路徑），只要 題目/學生答案/Promt 等資料夾與本檔案放在同一層即可直接執行。
# 若你的資料夾結構不同，請自行改成正確的相對或絕對路徑，例如 Path('./論文最終程式') 或 Path('C:/xxx/xxx')。
BASE_DIR = Path('.')
QUESTION_DIR = BASE_DIR / '題目'
ANSWER_DIR = BASE_DIR / '學生答案'
PROMPT_DIR = BASE_DIR / 'Promt'
OUTPUT_ROOT_DIR = BASE_DIR / '評分結果'

# 請填入要批改的學生答案檔名（不含 .txt），須對應 學生答案/ 資料夾內實際存在的檔案，例如 'C60V2-1'
STUDENTS = ['範例學生1', '範例學生2']

PROMPT_FILES = {
    'slm1': {'filename': 'SLM1.py', 'function': 'get_slm1_prompt'},
    'slm2': {'filename': 'SLM2.py', 'function': 'get_slm2_prompt'},
    'slm3': {'filename': 'SLM3.py', 'function': 'get_slm3_prompt'},
    'arbitration': {'filename': 'LLM仲裁.py', 'function': 'generate_prompt'},
}

QUESTION_FILES = {
    'behavioral': 'behavioral_layer.json',
    'specification': 'specification_layer.json',
    'syntax': 'syntax_layer.json',
}

LAYER_TO_SLM = {'behavioral': 'slm1', 'specification': 'slm2', 'syntax': 'slm3'}

MODELS = {
    'slm1': {'model': 'gemma4:latest', 'temperature': 0.0},
    'slm2': {'model': 'gemma4:latest', 'temperature': 0.0},
    'slm3': {'model': 'gemma4:latest', 'temperature': 0.0},
    'claude': {'model': 'claude-sonnet-5', 'temperature': 0.0},
}

# 同一個模型自己最多同時處理幾個請求，對應 Colab 端 Ollama 的 OLLAMA_NUM_PARALLEL 設定
# 三個模型彼此「嚴格分階段」執行（不會同時呼叫兩個不同模型），避免 GPU 內反覆換模型的開銷
REQUESTS_PER_MODEL = 3

MAX_TOKENS_ARBITRATION = 4096

## 連接 Colab 端 Ollama 服務

三個本地 SLM（qwen2.5-coder / gemma4 / phi4）實際執行於 Colab，並透過 ngrok 建立公開 Endpoint。請將 Colab 端印出的 `OLLAMA_API_CONFIG['endpoint']`（已包含 `/api/generate`）貼到下方輸入框。

In [ ]:
print('請輸入 Colab Ollama Endpoint（ngrok 產生、需含 /api/generate 的完整網址）:\n')
COLAB_ENDPOINT = input('Endpoint: ').strip()

if not COLAB_ENDPOINT:
    raise ValueError('Endpoint 不能為空')

print(f'\n已設置 Endpoint：{COLAB_ENDPOINT}')

OLLAMA_API_CONFIG = {
    'endpoint': COLAB_ENDPOINT,
    'timeout': 300,
    'retry_times': 3,
}

## 設定 Anthropic API Key

第四階段的 LLM 仲裁需要呼叫 Claude，因此需要一組有效的 Anthropic API Key。**金鑰絕不寫死在程式碼中、也不會存成任何檔案**，請在下方輸入框手動貼上（畫面與輸出皆不會顯示內容）。尚未申請的話可至 [console.anthropic.com](https://console.anthropic.com/) 取得。

若跳過此步驟或輸入錯誤金鑰，仲裁階段會自動走 Fallback（取 Round1 與 Round2 中較低分數），不會中斷整體流程，但建議務必設定正確金鑰以確保仲裁品質。

In [ ]:
print('請輸入 Anthropic API Key（輸入內容不會顯示於畫面，貼上後按 Enter 即可）:\n')
ANTHROPIC_API_KEY = getpass.getpass('ANTHROPIC_API_KEY: ').strip()

if not ANTHROPIC_API_KEY:
    raise ValueError('ANTHROPIC_API_KEY 不能為空')

print('\n已設置 Anthropic API Key（已隱藏顯示）')

## 階段一：前處理與防錯切分（Preprocessing）

- `PromptLoader`：動態載入 `Promt/` 資料夾內的 4 份 `.py` Prompt 定義檔，編譯為可呼叫函式。
- `QuestionRepository`：讀取並合併三層題目 JSON（behavioral / specification / syntax），以題號為單位整合成單一資料結構。
- `StudentAnswerParser`：以正規表達式 `\n*(?=Q\d+[:：]\n*)` 切分學生答案，避免 C# 原始碼中的大寫 `Q`（如 `Console.ReadLine`）誤觸發切分。

In [ ]:
class PromptLoader:
    def __init__(self, prompt_dir: Path, prompt_files: Dict[str, Dict[str, str]]):
        self.prompt_dir = prompt_dir
        self.prompt_files = prompt_files
        self._functions: Dict[str, Any] = {}
        self._load_all()

    def _load_module(self, py_path: Path):
        spec = importlib.util.spec_from_file_location(py_path.stem, py_path)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        return module

    def _load_all(self):
        for key, meta in self.prompt_files.items():
            py_path = self.prompt_dir / meta['filename']
            if not py_path.exists():
                raise FileNotFoundError(f'找不到 Prompt 檔案：{py_path}')
            module = self._load_module(py_path)
            func_name = meta['function']
            if not hasattr(module, func_name):
                raise AttributeError(f'{py_path.name} 內找不到函式 {func_name}')
            self._functions[key] = getattr(module, func_name)
            logger.info(f'Prompt 動態載入成功：{key} -> {py_path.name}::{func_name}')

    def get(self, key: str):
        return self._functions[key]


prompt_loader = PromptLoader(PROMPT_DIR, PROMPT_FILES)

In [ ]:
class QuestionRepository:
    def __init__(self, question_dir: Path, question_files: Dict[str, str]):
        self.question_dir = question_dir
        self.question_files = question_files
        self.questions: Dict[str, Dict[str, Any]] = {}
        self._load_and_merge()

    def _load_json(self, filename: str) -> Dict[str, Any]:
        path = self.question_dir / filename
        if not path.exists():
            raise FileNotFoundError(f'找不到題目檔案：{path}')
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)

    def _load_and_merge(self):
        raw = {layer: self._load_json(fname) for layer, fname in self.question_files.items()}
        base_layer = 'behavioral'
        question_ids = list(raw[base_layer].keys())
        for q_id in question_ids:
            base = raw[base_layer][q_id]
            merged = {
                'type': base['type'],
                'points': base['points'],
                'question_category': base['question_category'],
                'answer_type': base['answer_type'],
                'scoring_config': base['scoring_config'],
                'keywords': base['keywords'],
                'layers': {},
            }
            for layer in self.question_files.keys():
                layer_key = f'{layer}_layer'
                merged['layers'][layer] = raw[layer][q_id][layer_key]
            self.questions[q_id] = merged
        logger.info(f'題目載入完成，共 {len(self.questions)} 題，並完成三層合併。')

    def get(self, q_id: str) -> Dict[str, Any]:
        return self.questions[q_id]

    def all_ids(self) -> List[str]:
        return sorted(self.questions.keys(), key=lambda x: int(x[1:]))


question_repo = QuestionRepository(QUESTION_DIR, QUESTION_FILES)

In [ ]:
class StudentAnswerParser:
    SPLIT_PATTERN = re.compile(r'\n*(?=Q\d+[:：]\n*)')
    HEADER_PATTERN = re.compile(r'^Q(\d+)[:：]\s*')

    def __init__(self, answer_dir: Path):
        self.answer_dir = answer_dir

    def parse(self, student_name: str) -> Dict[str, str]:
        path = self.answer_dir / f'{student_name}.txt'
        if not path.exists():
            raise FileNotFoundError(f'找不到學生答案檔案：{path}')
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read()
        chunks = self.SPLIT_PATTERN.split(text)
        answers: Dict[str, str] = {}
        for chunk in chunks:
            chunk = chunk.strip('\n')
            if not chunk.strip():
                continue
            match = self.HEADER_PATTERN.match(chunk)
            if not match:
                continue
            q_id = f'Q{match.group(1)}'
            code = chunk[match.end():].strip('\n')
            answers[q_id] = code
        logger.info(f'學生 {student_name} 答案切分完成，共 {len(answers)} 題。')
        return answers


answer_parser = StudentAnswerParser(ANSWER_DIR)

## 階段二：批改雙次執行（SLMs Execution）

- `OllamaClient`：呼叫 Colab 端 Ollama REST API，內建重試與逾時保護，並用 `re.search(r'\{[\s\S]*\}')` 容錯擷取模型回覆中的 JSON；解析失敗時會先嘗試救援式提取，救不到才重試。
- `ClaudeArbitrator`：封裝官方 `anthropic` SDK，供第四階段仲裁使用；若未設定有效金鑰則直接停用、由 Fallback 接手。
- `SLMExecutor`：**嚴格分三階段、鎖定一個模型跑完才換下一個**，避免 Ollama 在 GPU 內反覆換模型（每次換模型清空+重新載入權重約 10~30 秒）。階段一鎖定 slm1，連續跑完 Round1 的 10 題、再連續跑完 Round2 的 10 題；階段二換 slm2 重複同樣流程；階段三換 slm3。同一模型內部的 20 次請求以 `REQUESTS_PER_MODEL`（對應 `OLLAMA_NUM_PARALLEL`）併發處理。若該層 `total_criteria = 0`（如語言題的行為層），則不呼叫模型，直接回傳 `0/0`。

In [ ]:
JSON_BLOCK_PATTERN = re.compile(r'\{[\s\S]*\}')
TRAILING_COMMA_PATTERN = re.compile(r',(\s*[}\]])')


def _strip_trailing_commas(text: str) -> str:
    return TRAILING_COMMA_PATTERN.sub(r'\1', text)


def extract_json(text: str) -> Optional[dict]:
    match = JSON_BLOCK_PATTERN.search(text)
    if not match:
        return None
    candidate = match.group(0)

    for variant in (candidate, _strip_trailing_commas(candidate)):
        for strict in (True, False):
            try:
                return json.loads(variant, strict=strict)
            except json.JSONDecodeError:
                continue
    return None


# 記錄「已知不支援 temperature 參數」的模型名稱（僅供 ClaudeArbitrator 仲裁呼叫使用；
# 本地 SLM 走 OllamaClient 的 options.temperature，不受此問題影響）。
_TEMPERATURE_UNSUPPORTED_MODELS: set = set()


def _is_temperature_unsupported_error(error: Exception) -> bool:
    message = str(error)
    return 'temperature' in message and 'deprecated' in message


def _extract_response_text(response) -> str:
    """具備 Extended Thinking 的模型，content 陣列可能含 ThinkingBlock，
    真正的文字答案不一定在 content[0]，需要找到 type == 'text' 的區塊。"""
    for block in response.content:
        if getattr(block, 'type', None) == 'text':
            return block.text
    return ''


class OllamaClient:
    def __init__(self, api_config: Dict[str, Any]):
        self.endpoint = api_config['endpoint']
        self.timeout = api_config.get('timeout', 300)
        self.retry_times = api_config.get('retry_times', 3)

    def generate(self, model: str, prompt: str, temperature: float, salvage_fn: Optional[Any] = None) -> Dict[str, Any]:
        payload = {
            'model': model,
            'prompt': prompt,
            'stream': False,
            'options': {'temperature': temperature},
        }
        last_error = None
        last_raw_text = ''
        for attempt in range(1, self.retry_times + 1):
            try:
                resp = requests.post(self.endpoint, json=payload, timeout=self.timeout)
                resp.raise_for_status()
                raw_text = resp.json().get('response', '')
                last_raw_text = raw_text
                parsed = extract_json(raw_text)
                if parsed is None:
                    if salvage_fn:
                        salvaged = salvage_fn(raw_text)
                        if salvaged:
                            logger.warning(f'呼叫 {model} JSON 解析失敗，救援式提取成功，跳過剩餘重試。')
                            return {'success': True, 'raw_text': raw_text, 'parsed': salvaged, 'salvaged': True}
                    raise ValueError(f'無法從模型回覆中解析出合法 JSON（原始回覆前200字：{raw_text[:200]}）')
                return {'success': True, 'raw_text': raw_text, 'parsed': parsed, 'salvaged': False}
            except Exception as e:
                last_error = e
                logger.warning(f'呼叫 {model} 第 {attempt}/{self.retry_times} 次失敗：{e}')
                time.sleep(2 * attempt)
        return {'success': False, 'raw_text': last_raw_text, 'parsed': None, 'error': str(last_error)}


ollama_client = OllamaClient(OLLAMA_API_CONFIG)

In [ ]:
class ClaudeArbitrator:
    def __init__(self, api_key: str, model_config: Dict[str, Any]):
        self.model = model_config['model']
        self.temperature = model_config['temperature']
        self.client = None
        if api_key and api_key != 'your_api_key_here':
            self.client = Anthropic(api_key=api_key)
        else:
            logger.warning('尚未設定有效的 ANTHROPIC_API_KEY，仲裁階段將直接走 Fallback。')

    def arbitrate(self, prompt: str, max_tokens: int = 1024) -> Dict[str, Any]:
        if self.client is None:
            return {'success': False, 'error': 'ANTHROPIC_API_KEY 未設定'}
        try:
            request_kwargs: Dict[str, Any] = {
                'model': self.model,
                'max_tokens': max_tokens,
                'messages': [{'role': 'user', 'content': prompt}],
            }
            if self.model not in _TEMPERATURE_UNSUPPORTED_MODELS:
                request_kwargs['temperature'] = self.temperature
            try:
                response = self.client.messages.create(**request_kwargs)
            except Exception as api_error:
                if _is_temperature_unsupported_error(api_error):
                    logger.warning(f'{self.model} 不支援 temperature 參數，記錄後之後呼叫此模型將自動略過。')
                    _TEMPERATURE_UNSUPPORTED_MODELS.add(self.model)
                    request_kwargs.pop('temperature', None)
                    response = self.client.messages.create(**request_kwargs)
                else:
                    raise
            token_usage = {'input': response.usage.input_tokens, 'output': response.usage.output_tokens}
            raw_text = _extract_response_text(response)
            parsed = extract_json(raw_text) if raw_text else None
            if parsed is None:
                return {'success': False, 'raw_text': raw_text, 'error': '無法解析 Claude 回覆中的 JSON', 'token_usage': token_usage}
            return {'success': True, 'raw_text': raw_text, 'parsed': parsed, 'token_usage': token_usage}
        except Exception as e:
            logger.error(f'Claude 仲裁呼叫失敗：{e}')
            return {'success': False, 'error': str(e)}


claude_arbitrator = ClaudeArbitrator(ANTHROPIC_API_KEY, MODELS['claude'])

In [ ]:
def build_layer_payload(q_id: str, question: Dict[str, Any], layer: str) -> str:
    layer_data = question['layers'][layer]
    payload = {
        'question_id': q_id,
        'type': question['type'],
        'points': question['points'],
        'question_category': question['question_category'],
        'focus': layer_data.get('focus', ''),
        'total_criteria': layer_data.get('total_criteria', 0),
        'criteria': layer_data.get('criteria', {}),
        'keywords': question.get('keywords', []),
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


def build_student_answer_text(code: str) -> str:
    return f'學生 C# 答案：\n```csharp\n{code}\n```'


def salvage_layer_fields(text: str, layer: str, q_id: str) -> Optional[Dict[str, Any]]:
    score_field = f'{layer}_score'
    total_match = re.search(r'"total_criteria"\s*:\s*(\d+)', text)
    met_match = re.search(r'"met_criteria"\s*:\s*(\d+)', text)
    score_match = re.search(r'"' + re.escape(score_field) + r'"\s*:\s*"\s*(\d+)\s*/\s*(\d+)\s*"', text)
    if not (total_match and met_match and score_match):
        return None
    return {
        'question_id': q_id,
        'layer': layer,
        'analysis': '[救援式提取] 完整 JSON 解析失敗，改用正規表達式從原始回覆中直接萃取分數欄位。',
        'criteria_check': [],
        'total_criteria': int(total_match.group(1)),
        'met_criteria': int(met_match.group(1)),
        score_field: f'{score_match.group(1)}/{score_match.group(2)}',
    }


class SLMExecutor:
    def __init__(self, ollama_client: OllamaClient, prompt_loader: PromptLoader, models: Dict[str, Any], requests_per_model: int = 3):
        self.ollama_client = ollama_client
        self.prompt_loader = prompt_loader
        self.models = models
        self.requests_per_model = requests_per_model

    def _run_single_layer(self, slm_key: str, layer: str, q_id: str, question: Dict[str, Any], student_code: str) -> Dict[str, Any]:
        layer_data = question['layers'][layer]
        total_criteria = layer_data.get('total_criteria', 0)
        score_field = f'{layer}_score'

        if total_criteria == 0:
            return {
                'question_id': q_id,
                'layer': layer,
                'analysis': '此題該層無評分準則，不執行模型評分。',
                'criteria_check': [],
                'total_criteria': 0,
                'met_criteria': 0,
                score_field: '0/0',
            }

        prompt_fn = self.prompt_loader.get(slm_key)
        question_json_text = build_layer_payload(q_id, question, layer)
        student_answer_text = build_student_answer_text(student_code)
        prompt_text = prompt_fn(question_json_text, student_answer_text)

        model_cfg = self.models[slm_key]
        salvage_fn = lambda text: salvage_layer_fields(text, layer, q_id)
        outcome = self.ollama_client.generate(model_cfg['model'], prompt_text, model_cfg['temperature'], salvage_fn=salvage_fn)

        if not outcome['success']:
            logger.error(f'{q_id} [{layer}] SLM 呼叫最終失敗：{outcome.get("error")}')
            return {
                'question_id': q_id,
                'layer': layer,
                'analysis': f'[錯誤] SLM 呼叫失敗：{outcome.get("error")}',
                'raw_response': outcome.get('raw_text', ''),
                'criteria_check': [],
                'total_criteria': total_criteria,
                'met_criteria': 0,
                score_field: f'0/{total_criteria}',
                '_error': True,
            }

        parsed = outcome['parsed']
        parsed.setdefault('question_id', q_id)
        parsed.setdefault('layer', layer)
        if outcome.get('salvaged'):
            parsed['_salvaged'] = True
            parsed['raw_response'] = outcome.get('raw_text', '')
        return parsed

    def _run_model_phase(self, layer: str, slm_key: str, question_repo: QuestionRepository,
                          student_answers: Dict[str, str], round1_raw: Dict[str, Dict[str, Any]],
                          round2_raw: Dict[str, Dict[str, Any]]) -> None:
        q_ids = question_repo.all_ids()
        logger.info(f'=== 鎖定模型 {slm_key}（{layer}層）：開始連續處理 Round1 + Round2 共 {len(q_ids) * 2} 次提問 ===')

        with ThreadPoolExecutor(max_workers=self.requests_per_model) as pool:
            futures = {
                pool.submit(
                    self._run_single_layer, slm_key, layer, q_id,
                    question_repo.get(q_id), student_answers.get(q_id, '')
                ): q_id
                for q_id in q_ids
            }
            for future in as_completed(futures):
                q_id = futures[future]
                round1_raw[q_id][layer] = future.result()

        with ThreadPoolExecutor(max_workers=self.requests_per_model) as pool:
            futures = {
                pool.submit(
                    self._run_single_layer, slm_key, layer, q_id,
                    question_repo.get(q_id), student_answers.get(q_id, '')
                ): q_id
                for q_id in q_ids
            }
            for future in as_completed(futures):
                q_id = futures[future]
                round2_raw[q_id][layer] = future.result()

        logger.info(f'=== 模型 {slm_key} 已完成全部工作，換下一個模型 ===')

    def run_all(self, question_repo: QuestionRepository, student_answers: Dict[str, str]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        q_ids = question_repo.all_ids()
        for q_id in q_ids:
            if not student_answers.get(q_id):
                logger.warning(f'找不到 {q_id} 的學生作答，將以空字串送評。')

        round1_raw: Dict[str, Dict[str, Any]] = {q_id: {} for q_id in q_ids}
        round2_raw: Dict[str, Dict[str, Any]] = {q_id: {} for q_id in q_ids}

        # 嚴格依序鎖定 slm1 -> slm2 -> slm3，任何時刻只有一個模型在接收請求，避免 GPU 反覆換模型
        for layer, slm_key in LAYER_TO_SLM.items():
            self._run_model_phase(layer, slm_key, question_repo, student_answers, round1_raw, round2_raw)

        round1_all = {q_id: {layer: round1_raw[q_id][layer] for layer in LAYER_TO_SLM.keys()} for q_id in q_ids}
        round2_all = {q_id: {layer: round2_raw[q_id][layer] for layer in LAYER_TO_SLM.keys()} for q_id in q_ids}
        return round1_all, round2_all


slm_executor = SLMExecutor(ollama_client, prompt_loader, MODELS, requests_per_model=REQUESTS_PER_MODEL)

## 階段三：實質分數差異過濾（Divergence Analysis）

只比較正規化後的「實質分數」（`met_criteria`），忽略文字、空白、標點差異。當且僅當 Round1 與 Round2 的 `met_criteria` 不同時，該層才標記為分歧，並記錄雙方推理過程供仲裁使用。

In [ ]:
def parse_score_ratio(score_str: str) -> Tuple[int, int]:
    try:
        m, n = str(score_str).split('/')
        return int(m), int(n)
    except Exception:
        return (0, 0)


class DivergenceAnalyzer:
    def analyze(self, q_id: str, round1_layers: Dict[str, Any], round2_layers: Dict[str, Any]) -> Dict[str, Any]:
        divergent_layers: List[str] = []
        diff_details: Dict[str, Any] = {}
        for layer, slm_key in LAYER_TO_SLM.items():
            score_field = f'{layer}_score'
            r1 = round1_layers[layer]
            r2 = round2_layers[layer]
            r1_met, _ = parse_score_ratio(r1.get(score_field, '0/0'))
            r2_met, _ = parse_score_ratio(r2.get(score_field, '0/0'))
            if r1_met != r2_met:
                divergent_layers.append(slm_key)
                diff_details[slm_key] = {
                    'round1_score': r1.get(score_field),
                    'round1_analysis': r1.get('analysis', ''),
                    'round2_score': r2.get(score_field),
                    'round2_analysis': r2.get('analysis', ''),
                }
        return {
            'question_id': q_id,
            'divergent_layers': divergent_layers,
            'diff_details': diff_details,
        }


divergence_analyzer = DivergenceAnalyzer()

## 階段四：題為單位之 LLM 仲裁（Arbitration）

僅將有分歧的題目、僅針對「不一致層級」的準則與雙次推理過程，打包送給 Claude 仲裁，嚴格職責隔離（不可跨層批改），並最大化節省 Token。

In [ ]:
class ArbitrationEngine:
    def __init__(self, claude: ClaudeArbitrator, prompt_loader: PromptLoader):
        self.claude = claude
        self.prompt_loader = prompt_loader

    def _build_criteria_context(self, question: Dict[str, Any]) -> Dict[str, Any]:
        return {
            '行為層準則': question['layers']['behavioral'].get('criteria', {}),
            '規範層準則': question['layers']['specification'].get('criteria', {}),
            '語言層準則': question['layers']['syntax'].get('criteria', {}),
        }

    def arbitrate_question(self, q_id: str, question: Dict[str, Any], student_code: str, divergence: Dict[str, Any]) -> Tuple[Optional[Dict[str, Any]], bool]:
        arbitration_input = {
            '題號': q_id,
            '學生答案': student_code,
            '不一致層級': divergence['divergent_layers'],
            '差異詳情': divergence['diff_details'],
        }
        arbitration_input.update(self._build_criteria_context(question))

        prompt_fn = self.prompt_loader.get('arbitration')
        prompt_text = prompt_fn(arbitration_input)

        outcome = self.claude.arbitrate(prompt_text, max_tokens=MAX_TOKENS_ARBITRATION)
        if outcome.get('success'):
            parsed = outcome['parsed']
            if q_id in parsed and outcome.get('token_usage'):
                parsed[q_id]['_token_usage'] = outcome['token_usage']
            return parsed, True
        logger.warning(f'{q_id} Claude 仲裁失敗，將啟動 Fallback 保守原則：{outcome.get("error")}')
        return None, False


arbitration_engine = ArbitrationEngine(claude_arbitrator, prompt_loader)

## 階段五：分數整合、公式加權與退路（Finalization）

- 一致時採用 Round2 分數；分歧時採用 Claude 仲裁分數。
- **Robust Fallback**：若 Claude 仲裁失敗（斷線或例外），自動取 Round1 與 Round2 中「較低分數」作為最終得分，僅記錄警告、程式絕不中斷。
- 加權公式：
  `加權分數 = ((行為比率 × w_行為) + (語言比率 × w_語言)) × 規範係數 × 10`
  其中「規範係數」在 `weights.specification == "乘法係數"` 時，採用規範層實際得分比例；否則直接採用 JSON 內給定的數值常數（如語言題的 `1.0`）。
- `total_score` 為 10 題加權分數加總（滿分 100）。

In [ ]:
class FinalScorer:
    @staticmethod
    def _ratio(layer_result: Dict[str, Any], default: float = 0.0) -> float:
        total = layer_result.get('total_criteria', 0) or 0
        met = layer_result.get('met_criteria', 0) or 0
        if not total:
            return default
        return met / total

    def _resolve_final_layer(self, layer: str, slm_key: str, round2_layers: Dict[str, Any],
                              divergence: Dict[str, Any], arbitration_results: Dict[str, Any],
                              arbitration_ok: bool, q_id: str) -> Dict[str, Any]:
        score_field = f'{layer}_score'
        is_divergent = slm_key in divergence['divergent_layers']

        if not is_divergent:
            r2 = round2_layers[layer]
            met, total = parse_score_ratio(r2.get(score_field, '0/0'))
            return {'met_criteria': met, 'total_criteria': total, 'score': f'{met}/{total}', 'source': 'Round2一致採用'}

        if arbitration_ok and arbitration_results:
            layer_result = arbitration_results.get(q_id, {}).get(layer)
            if layer_result:
                met = layer_result.get('met_criteria', 0)
                total = layer_result.get('total_criteria', 0)
                return {
                    'met_criteria': met, 'total_criteria': total, 'score': f'{met}/{total}',
                    'reasoning': layer_result.get('reasoning', ''), 'source': 'Claude仲裁',
                }

        detail = divergence['diff_details'].get(slm_key, {})
        r1_met, r1_total = parse_score_ratio(detail.get('round1_score', '0/0'))
        r2_met, r2_total = parse_score_ratio(detail.get('round2_score', '0/0'))
        if r1_met <= r2_met:
            chosen_met, chosen_total = r1_met, r1_total
        else:
            chosen_met, chosen_total = r2_met, r2_total
        return {
            'met_criteria': chosen_met, 'total_criteria': chosen_total, 'score': f'{chosen_met}/{chosen_total}',
            'source': 'Fallback取Round1與Round2較低分',
        }

    def score_question(self, q_id: str, question: Dict[str, Any], round2_layers: Dict[str, Any],
                        divergence: Dict[str, Any], arbitration_results: Dict[str, Any],
                        arbitration_ok: bool) -> Dict[str, Any]:
        final_layers: Dict[str, Any] = {}
        for layer, slm_key in LAYER_TO_SLM.items():
            final_layers[layer] = self._resolve_final_layer(
                layer, slm_key, round2_layers, divergence, arbitration_results, arbitration_ok, q_id
            )

        weights = question['scoring_config']['weights']
        behavioral_ratio = self._ratio(final_layers['behavioral'])
        syntax_ratio = self._ratio(final_layers['syntax'])

        spec_weight = weights.get('specification')
        if spec_weight == '乘法係數':
            spec_coefficient = self._ratio(final_layers['specification'], default=1.0)
        else:
            spec_coefficient = float(spec_weight)

        w_behavioral = float(weights.get('behavioral', 0) or 0)
        w_syntax = float(weights.get('syntax', 0) or 0)

        weighted_score = ((behavioral_ratio * w_behavioral) + (syntax_ratio * w_syntax)) * spec_coefficient * 10

        return {
            'question_id': q_id,
            'type': question['type'],
            'question_category': question['question_category'],
            'final_layers': final_layers,
            'spec_coefficient': round(spec_coefficient, 4),
            'weighted_score': round(weighted_score, 3),
        }


final_scorer = FinalScorer()

## 主控 Pipeline：ExperimentRunner

大迴圈依序走訪 `STUDENTS`，每位學生自動建立 `評分結果/{student}/{timestamp}/` 輸出目錄，並落地 5 份 JSON 成果檔案。任一學生批改過程發生未預期錯誤時，記錄錯誤並繼續下一位，不中斷整體流程。

In [ ]:
class ExperimentRunner:
    def __init__(self):
        self.question_repo = question_repo
        self.prompt_loader = prompt_loader
        self.answer_parser = answer_parser
        self.slm_executor = slm_executor
        self.divergence_analyzer = divergence_analyzer
        self.arbitration_engine = arbitration_engine
        self.final_scorer = final_scorer

    def _make_output_dir(self, student_name: str) -> Path:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_dir = OUTPUT_ROOT_DIR / student_name / timestamp
        output_dir.mkdir(parents=True, exist_ok=True)
        return output_dir

    @staticmethod
    def _save_json(path: Path, data: Any):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    @staticmethod
    def _sum_run_tokens(round_all: Dict[str, Any]) -> Dict[str, int]:
        """加總一輪（Round1 或 Round2）的 token 用量。本地 SLM（Ollama）不計費也不記錄 token_usage，
        故此處只會加總到 Claude 等雲端呼叫留下的 token_usage（本組沒有，恆為 0）。"""
        total_input = 0
        total_output = 0
        for layers in round_all.values():
            for layer_result in layers.values():
                usage = layer_result.get('token_usage')
                if usage:
                    total_input += usage.get('input', 0)
                    total_output += usage.get('output', 0)
        return {'input': total_input, 'output': total_output}

    @staticmethod
    def _sum_arbitration_tokens(arbitration_results: Dict[str, Any]) -> Dict[str, int]:
        """加總仲裁階段的 token 用量：每題一次仲裁呼叫（可能同時涵蓋多個分歧層），每題只累加一次。"""
        total_input = 0
        total_output = 0
        for result in arbitration_results.values():
            usage = result.get('_token_usage')
            if usage:
                total_input += usage.get('input', 0)
                total_output += usage.get('output', 0)
        return {'input': total_input, 'output': total_output}

    def run_for_student(self, student_name: str) -> Dict[str, Any]:
        logger.info('=' * 60)
        logger.info(f'開始批改學生：{student_name}')
        logger.info('=' * 60)

        output_dir = self._make_output_dir(student_name)
        student_answers = self.answer_parser.parse(student_name)

        logger.info('--- 依序鎖定 slm1 -> slm2 -> slm3，各自連續跑完 Round1+Round2 ---')
        round1_all, round2_all = self.slm_executor.run_all(self.question_repo, student_answers)

        self._save_json(output_dir / 'round1.json', round1_all)
        self._save_json(output_dir / 'round2.json', round2_all)
        logger.info('round1.json / round2.json 已落地')

        divergence_report: Dict[str, Any] = {}
        divergent_count = 0
        for q_id in self.question_repo.all_ids():
            divergence = self.divergence_analyzer.analyze(q_id, round1_all[q_id], round2_all[q_id])
            if divergence['divergent_layers']:
                divergent_count += 1
                divergence_report[q_id] = divergence

        divergence_output = {'divergent_count': divergent_count, 'details': divergence_report}
        self._save_json(output_dir / '差異分析.json', divergence_output)
        logger.info(f'差異分析.json 已落地（分歧題數：{divergent_count}）')

        arbitration_results: Dict[str, Any] = {}
        arbitration_ok_map: Dict[str, bool] = {}
        for q_id, divergence in divergence_report.items():
            question = self.question_repo.get(q_id)
            student_code = student_answers.get(q_id, '')
            parsed, ok = self.arbitration_engine.arbitrate_question(q_id, question, student_code, divergence)
            arbitration_ok_map[q_id] = ok
            if ok and parsed:
                arbitration_results.update(parsed)

        self._save_json(output_dir / 'llm_仲裁結果.json', arbitration_results)
        logger.info('llm_仲裁結果.json 已落地')

        final_report: Dict[str, Any] = {}
        total_score = 0.0
        for q_id in self.question_repo.all_ids():
            question = self.question_repo.get(q_id)
            divergence = divergence_report.get(q_id, {'divergent_layers': [], 'diff_details': {}})
            ok = arbitration_ok_map.get(q_id, False)
            scored = self.final_scorer.score_question(
                q_id, question, round2_all[q_id], divergence, arbitration_results, ok
            )
            final_report[q_id] = scored
            total_score += scored['weighted_score']

        round1_tokens = self._sum_run_tokens(round1_all)
        round2_tokens = self._sum_run_tokens(round2_all)
        arbitration_tokens = self._sum_arbitration_tokens(arbitration_results)
        token_usage_summary = {
            'round1': round1_tokens,
            'round2': round2_tokens,
            'arbitration': arbitration_tokens,
            'grand_total': {
                'input': round1_tokens['input'] + round2_tokens['input'] + arbitration_tokens['input'],
                'output': round1_tokens['output'] + round2_tokens['output'] + arbitration_tokens['output'],
            },
        }
        token_usage_summary['grand_total']['total'] = (
            token_usage_summary['grand_total']['input'] + token_usage_summary['grand_total']['output']
        )

        final_output = {
            'student': student_name,
            'timestamp': output_dir.name,
            'questions': final_report,
            'total_score': round(total_score, 2),
            'token_usage': token_usage_summary,
        }
        self._save_json(output_dir / '最終評分.json', final_output)
        logger.info(
            f'最終評分.json 已落地（總分：{final_output["total_score"]}，'
            f'總token數：{token_usage_summary["grand_total"]["total"]}）'
        )

        return final_output

    def run_all(self) -> Dict[str, Any]:
        summary: Dict[str, Any] = {}
        for student_name in STUDENTS:
            try:
                summary[student_name] = self.run_for_student(student_name)
            except Exception as e:
                logger.error(f'學生 {student_name} 批改流程發生未預期錯誤，已跳過並繼續下一位：{e}', exc_info=True)
                summary[student_name] = {'error': str(e)}
        return summary

## 執行大迴圈（依 ROUNDS 設定執行）

一輪＝完整跑完一次 Round1+Round2（含差異比對、仲裁、加權，`ExperimentRunner.run_all()` 內部已經包含完整雙次）。這裡會依照 `ROUNDS` 自動連續執行對應輪數，每輪各自落地到獨立的 `{timestamp}` 資料夾，彼此不會互相覆蓋。單一輪若發生未預期錯誤會記錄下來並跳過，繼續下一輪，不中斷整體流程。

In [ ]:
ROUNDS = 1
round_results: List[Dict[str, Any]] = []

for round_idx in range(1, ROUNDS + 1):
    logger.info('#' * 60)
    logger.info(f'開始第 {round_idx}/{ROUNDS} 輪（Round1+Round2 算一輪）')
    logger.info('#' * 60)
    try:
        runner = ExperimentRunner()
        results_summary = runner.run_all()
        round_results.append({'round': round_idx, 'results': results_summary})

        print(f'\n第 {round_idx}/{ROUNDS} 輪完成！')
        for student, result in results_summary.items():
            if 'error' in result:
                print(f'  [失敗] {student}: {result["error"]}')
            else:
                print(f'  [完成] {student}: 總分 {result["total_score"]}')
    except Exception as e:
        logger.error(f'第 {round_idx}/{ROUNDS} 輪發生未預期錯誤，記錄後繼續下一輪：{e}', exc_info=True)
        round_results.append({'round': round_idx, 'results': {'error': str(e)}})

print(f'\n全部 {ROUNDS} 輪執行完成！')